In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_store")

In [0]:
display(df)

In [0]:
row_count = df.count()
row_count

In [0]:
# DUPLICATE ANALYSIS
print("\nDUPLICATE ANALYSIS")
duplicate_store_ids = df.groupBy("store_id").count().filter(col("count") > 1)
print(f"Duplicate store_ids: {duplicate_store_ids.count()}")

duplicate_names = df.groupBy("store_name").count().filter(col("count") > 1)
print(f"Duplicate store_names: {duplicate_names.count()}")


In [0]:
#NULL VALUE ANALYSIS
print("\nNULL VALUE ANALYSIS")

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
])
print("Null counts by column:")
display(null_counts)

In [0]:
# Fill null manager_name using manager_id from other rows
from pyspark.sql.window import Window

window_spec = Window.partitionBy("manager_id").orderBy(f.col("manager_name").desc())
df = df.withColumn(
    "manager_name",
    f.first("manager_name", ignorenulls=True).over(window_spec)
)

display(df.orderBy("store_id"))

In [0]:
# Check store_type values
print("\nStore_type distribution:")
df.groupBy("store_type").count().orderBy("count", ascending=False).show()

In [0]:
# Check for whitespace issues
print("\nChecking for whitespace issues in string columns")
for col_name in ["store_name", "city", "state", "manager_id", "manager_name", "store_type"]:
    whitespace_count = df.filter((col(col_name) != f.trim(col(col_name))) | (col(col_name).rlike("  "))  )

In [0]:
# State code validation
print("\nState/Territory codes found:")
df.groupBy("state").count().orderBy("state").show(100, truncate=False)


In [0]:
# 5. SUMMARY STATISTICS
print("SUMMARY STATISTICS")
print(f"Total records: {row_count}")
print(f"Distinct store_ids: {df.select('store_id').distinct().count()}")
print(f"Distinct managers: {df.select('manager_id').distinct().count()}")
print(f"Year range: {df.agg(f.min('opened_year'), f.max('opened_year')).collect()[0]}")

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_store")